# EDA — Jigsaw Toxic Comment Classification

Person B, Week 1 catch-up. Run after downloading `train.csv`, `test.csv` and `test_labels.csv` into `data/raw/`.

Answers here feed `docs/decisions.md` (D1–D4). Never commit anything under `data/`.


In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
RAW = ROOT / "data" / "raw"
print("RAW:", RAW, "exists:", RAW.exists())
print("files:", [p.name for p in RAW.glob("*.csv")] if RAW.exists() else [])


In [ ]:
train = pd.read_csv(RAW / "train.csv")
test = pd.read_csv(RAW / "test.csv")
test_labels = pd.read_csv(RAW / "test_labels.csv")

print("train:", train.shape)
print("test:", test.shape)
print("test_labels:", test_labels.shape)
train.head(3)


## 1. Which rows are actually scored? (decision D1)

In `test_labels.csv`, a value of `-1` means the row was not scored. Count those rows.


In [ ]:
LABELS = ["toxic", "severe_toxic", "obscene", "threat", "insult", "identity_hate"]

unscored = (test_labels[LABELS] == -1).any(axis=1)
scored = test_labels[~unscored].copy()
print("rows with any -1:", int(unscored.sum()))
print("scored rows available:", len(scored))
print("scored toxic %:", round((scored[LABELS].max(axis=1) > 0).mean() * 100, 2))


## 2. Class balance (train.csv)

Binary label = `1` if any of the 6 columns is 1. Accuracy is a bad metric here.


In [ ]:
train["label"] = (train[LABELS].max(axis=1) > 0).astype(int)
print("toxic rows:", int(train["label"].sum()))
print("toxic %:", round(train["label"].mean() * 100, 2))
print("per-label counts:")
print(train[LABELS].sum().sort_values(ascending=False))


## 3. Comment length distribution and truncation risk

The model truncates at 512 tokens. Measure how long comments are in characters as a proxy.


In [ ]:
lengths = train["comment_text"].astype(str).str.len()
print(lengths.describe())
plt.figure(figsize=(8, 4))
plt.hist(lengths.clip(upper=2000), bins=50)
plt.xlabel("characters (capped at 2000)")
plt.ylabel("count")
plt.title("Comment length distribution")
plt.show()


## 4. Label overlap (multi-label check)


In [ ]:
train["n_labels"] = train[LABELS].sum(axis=1)
print(train["n_labels"].value_counts().sort_index())
print("rows with >1 label:", int((train["n_labels"] > 1).sum()))


## 5. Example rows (mask offensive words before sharing)

Keep examples few and masked in the report — the dataset contains slurs and threats.


In [ ]:
def short(text: str, limit: int = 300) -> str:
    return " ".join(str(text).split())[:limit]

for name, value in [("non_toxic", 0), ("toxic", 1)]:
    print(f"\n--- {name} examples ---")
    for text in train.loc[train["label"] == value, "comment_text"].head(10):
        print("-", short(text))


## Findings to copy into notes / the report

Filled from the raw Kaggle CSVs on 27 Nov 2026 (local run).

- 	rain.csv rows: **159,571**
- Toxic % in train: **10.17%** (binary = any of 6 labels)
- Per-label counts: toxic 15,294; obscene 8,449; insult 7,877; severe_toxic 1,595; identity_hate 1,405; threat 478
- Rows in 	est_labels.csv with -1: **89,186**
- Scored rows available for val+test: **63,978** (toxic 9.76%)
- Split produced: val 3,000 / test 3,000, both toxic 9.77%, seed 42, reproducible (SHA256 verified)
- Most common label: **toxic** (15,294)
- Rows with more than one label: **9,865**
- Comments over 2,000 chars (truncation proxy): **4,049** - token-level count still TODO
- Decision D1 answer: use official test set? **Yes** - drop 89,186 unscored rows, 63,978 remain
- Decision D4 answer: 	oxic only or max-of-6? ___ (Week 4)
